# **DEPLOYMENT PREP**

**We are going to Save Tweedie Model + Feature Pipeline**

# 🚀 Deployment Preparation

This notebook prepares the final Tweedie XGBoost model for deployment.  
The goal is to produce a **reproducible, portable inference pipeline** that can be used in:

- batch forecasting jobs  
- pricing/markdown simulations  
- inventory replenishment systems  
- SKU-level demand dashboards  

### What we save:
1. **Trained Tweedie model**  
   Stored as `model`.

2. **Feature list**  
   Ensures inference uses the exact same features as training.

3. **Category mappings**  
   If categorical variables were encoded, we save the mapping so inference is consistent.

These artifacts allow us to build a stable scoring function and deploy the model in any environment (Python script, API, Airflow job, etc.).


In [2]:
import joblib
import xgboost as xgb
import numpy as np
import pandas as pd

# load artifacts saved from Notebook 5
model = joblib.load("models/tweedie_xgb_model.pkl")
FEATURES = joblib.load("models/feature_list.pkl")

print("Loaded Tweedie model and feature list.")


Loaded Tweedie model and feature list.


## 🔮 Inference Function (Single-Row Scoring)

The `score_row()` function provides a clean, production-ready way to generate
predictions for a single SKU-day input. This is useful for:

- real-time dashboards  
- API endpoints  
- simulation tools  
- interactive notebooks  

The function ensures feature consistency by:
- adding missing features with default values  
- applying the exact feature order used during training  
- clipping negative predictions to maintain valid count outputs  

This is the core building block for deployment.


In [4]:
# Notebook 7 — Cell 2: Inference Function

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib

# model and FEATURES were loaded in Cell 1
# model = joblib.load("models/tweedie_xgb_model.pkl")
# FEATURES = joblib.load("models/feature_list.pkl")

def score_row(row_dict):
    """
    Score a single SKU-day row using the saved Tweedie model.
    row_dict: dictionary of feature_name → value
    """
    # Convert dict → DataFrame
    df_row = pd.DataFrame([row_dict])

    # Ensure all required features exist
    missing = [f for f in FEATURES if f not in df_row.columns]
    for m in missing:
        df_row[m] = 0

    # Build DMatrix
    dmatrix = xgb.DMatrix(df_row[FEATURES])

    # Predict
    pred = model.predict(dmatrix)

    # Clip negative predictions
    return float(np.clip(pred[0], 0, None))

print("Inference function ready.")


Inference function ready.


## 📦 Batch Scoring Function

The `score_batch()` function allows the model to score thousands of SKU-day rows
at once. This is the production-ready method used for:

- daily demand forecasting
- weekly replenishment planning
- markdown simulations
- inventory optimization workflows

The function ensures:
- all required features are present
- the feature order matches training
- predictions are clipped to valid non-negative values

This is the core of the batch inference pipeline used in deployment.


In [5]:
# Notebook 7 — Cell 3: Batch Prediction

def score_batch(df):
    """
    Score an entire DataFrame of SKU-day rows using the saved Tweedie model.
    df: DataFrame containing all required features.
    Returns the same DataFrame with a new column: predicted_units.
    """

    # Ensure all required features exist
    for f in FEATURES:
        if f not in df.columns:
            df[f] = 0

    # Build DMatrix
    dmatrix = xgb.DMatrix(df[FEATURES])

    # Predict
    preds = model.predict(dmatrix)

    # Clip negative predictions
    preds = np.clip(preds, 0, None)

    # Add predictions to DataFrame
    df["predicted_units"] = preds

    return df

print("Batch scoring function ready.")


Batch scoring function ready.
